# Customer Churn Predictor - Business ML Application
## Module 2, Session 3: Predicting Customer Churn for Telecom Company

**Goal:** Predict which customers will leave (churn) your telecom service  
**Dataset:** Telecom customer data with 19 features  
**Problem:** Binary classification with business impact  
**Target Accuracy:** >80%  

---

### What You'll Learn
1. Real-world business ML application
2. Feature engineering for categorical variables
3. Label Encoding vs One-Hot Encoding
4. Random Forest classifier
5. Feature importance analysis
6. Business impact calculation
7. Cost of false negatives vs false positives

---

**Created:** 2026-01-06  
**Course:** ML for Engineers  
**Module:** 2 - Classification

## Step 1: Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)

print("✓ All libraries imported successfully!")

## Step 2: Create Synthetic Telecom Customer Dataset

We'll create a realistic telecom churn dataset with customer features.

> **AI Assistant Prompt:**  
> "Create a synthetic telecom customer churn dataset with 1000 customers. Include features like tenure (months), monthly charges, contract type (month-to-month, 1-year, 2-year), internet service, tech support, and churn label. Make churn correlate with short tenure and month-to-month contracts."

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Number of customers
n_customers = 1000

# Generate customer data
data = {
    'customer_id': [f'CUST{i:04d}' for i in range(n_customers)],
    'gender': np.random.choice(['Male', 'Female'], n_customers),
    'senior_citizen': np.random.choice([0, 1], n_customers, p=[0.85, 0.15]),
    'partner': np.random.choice(['Yes', 'No'], n_customers, p=[0.5, 0.5]),
    'dependents': np.random.choice(['Yes', 'No'], n_customers, p=[0.3, 0.7]),
    'tenure_months': np.random.randint(1, 73, n_customers),
    'phone_service': np.random.choice(['Yes', 'No'], n_customers, p=[0.9, 0.1]),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], n_customers, p=[0.35, 0.45, 0.2]),
    'online_security': np.random.choice(['Yes', 'No', 'No internet'], n_customers, p=[0.3, 0.5, 0.2]),
    'tech_support': np.random.choice(['Yes', 'No', 'No internet'], n_customers, p=[0.3, 0.5, 0.2]),
    'contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_customers, p=[0.55, 0.25, 0.2]),
    'paperless_billing': np.random.choice(['Yes', 'No'], n_customers, p=[0.6, 0.4]),
    'payment_method': np.random.choice(
        ['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], 
        n_customers, 
        p=[0.35, 0.2, 0.25, 0.2]
    ),
}

# Monthly charges (depends on services)
base_charge = 20
internet_charge = np.where(
    np.array([data['internet_service']]) == 'Fiber optic', 30,
    np.where(np.array([data['internet_service']]) == 'DSL', 15, 0)
)[0]
data['monthly_charges'] = base_charge + internet_charge + np.random.uniform(-5, 15, n_customers)
data['monthly_charges'] = np.round(data['monthly_charges'], 2)

# Total charges (tenure * monthly charges with some variation)
data['total_charges'] = data['tenure_months'] * data['monthly_charges'] + np.random.uniform(-50, 50, n_customers)
data['total_charges'] = np.round(data['total_charges'], 2)
data['total_charges'] = np.maximum(data['total_charges'], data['monthly_charges'])  # At least one month

# Create DataFrame
df = pd.DataFrame(data)

# Generate churn based on logical patterns
churn_probability = np.zeros(n_customers)

# Higher churn for:
# - Short tenure
churn_probability += np.where(df['tenure_months'] < 12, 0.4, 0.1)
# - Month-to-month contracts
churn_probability += np.where(df['contract'] == 'Month-to-month', 0.3, 0.0)
# - No tech support
churn_probability += np.where(df['tech_support'] == 'No', 0.15, 0.0)
# - Electronic check payment (hassle)
churn_probability += np.where(df['payment_method'] == 'Electronic check', 0.1, 0.0)
# - Senior citizens
churn_probability += np.where(df['senior_citizen'] == 1, 0.05, 0.0)
# - No partner
churn_probability += np.where(df['partner'] == 'No', 0.05, 0.0)

# Generate churn
df['churn'] = np.random.binomial(1, np.clip(churn_probability, 0, 0.9))
df['churn_label'] = df['churn'].map({0: 'No', 1: 'Yes'})

print("✓ Synthetic telecom dataset created!")
print(f"\nDataset shape: {df.shape}")
print(f"Total customers: {len(df)}")

In [ ]:
# Display first few rows
print("Sample Customer Data:")
display(df.head(10))

# Dataset info
print("\nDataset Info:")
print(df.info())

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Churn distribution
print("Churn Distribution:")
print(df['churn_label'].value_counts())
print(f"\nChurn rate: {df['churn'].mean()*100:.1f}%")

# Visualize churn distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
df['churn_label'].value_counts().plot(
    kind='bar', 
    ax=axes[0], 
    color=['#4ECDC4', '#FF6B6B'], 
    alpha=0.8, 
    edgecolor='black'
)
axes[0].set_title('Customer Churn Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_xticklabels(['No Churn', 'Churned'], rotation=0)
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
colors = ['#4ECDC4', '#FF6B6B']
df['churn_label'].value_counts().plot(
    kind='pie', 
    ax=axes[1], 
    colors=colors, 
    autopct='%1.1f%%',
    startangle=90
)
axes[1].set_title('Churn Percentage', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

if df['churn'].mean() < 0.5:
    print("\n⚠️ This is an IMBALANCED dataset - more customers stay than leave.")
else:
    print("\n✓ Relatively balanced dataset.")

In [ ]:
# Analyze churn by contract type
print("Churn Rate by Contract Type:")
contract_churn = df.groupby('contract')['churn'].mean().sort_values(ascending=False)
print(contract_churn * 100)

# Visualize
plt.figure(figsize=(10, 5))
(contract_churn * 100).plot(
    kind='bar', 
    color='#FF6B6B', 
    alpha=0.8, 
    edgecolor='black'
)
plt.title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
plt.xlabel('Contract Type', fontsize=12)
plt.ylabel('Churn Rate (%)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Month-to-month customers churn WAY more!")

In [ ]:
# Analyze tenure vs churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tenure distribution by churn
df[df['churn']==0]['tenure_months'].hist(
    bins=30, alpha=0.6, label='No Churn', color='#4ECDC4', ax=axes[0]
)
df[df['churn']==1]['tenure_months'].hist(
    bins=30, alpha=0.6, label='Churned', color='#FF6B6B', ax=axes[0]
)
axes[0].set_xlabel('Tenure (months)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Tenure Distribution by Churn', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Monthly charges distribution by churn
df[df['churn']==0]['monthly_charges'].hist(
    bins=30, alpha=0.6, label='No Churn', color='#4ECDC4', ax=axes[1]
)
df[df['churn']==1]['monthly_charges'].hist(
    bins=30, alpha=0.6, label='Churned', color='#FF6B6B', ax=axes[1]
)
axes[1].set_xlabel('Monthly Charges ($)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Monthly Charges by Churn', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 New customers (low tenure) are more likely to churn!")

## Step 4: Feature Engineering - Handle Categorical Variables

Machine learning models need numbers, not text categories!

**Two approaches:**
1. **Label Encoding:** Convert categories to numbers (0, 1, 2...)
2. **One-Hot Encoding:** Create binary columns for each category

> **AI Assistant Prompt:**  
> "Explain the difference between Label Encoding and One-Hot Encoding. Show how to apply Label Encoding to binary categorical features (Yes/No) and create a complete preprocessing pipeline for the churn dataset."

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

print("Original categorical columns:")
categorical_cols = df_processed.select_dtypes(include=['object']).columns
print(list(categorical_cols))

print("\n" + "="*60)
print("FEATURE ENGINEERING")
print("="*60)

In [ ]:
# Drop customer_id (not useful for prediction)
df_processed = df_processed.drop('customer_id', axis=1)

# Binary encoding for Yes/No columns
binary_cols = ['partner', 'dependents', 'phone_service', 'paperless_billing']
for col in binary_cols:
    df_processed[col] = df_processed[col].map({'Yes': 1, 'No': 0})

print("✓ Binary columns encoded (Yes=1, No=0)")

# Label encoding for gender
df_processed['gender'] = df_processed['gender'].map({'Male': 1, 'Female': 0})
print("✓ Gender encoded (Male=1, Female=0)")

# One-hot encoding for multi-category columns
# Contract type
contract_dummies = pd.get_dummies(df_processed['contract'], prefix='contract')
df_processed = pd.concat([df_processed, contract_dummies], axis=1)
df_processed = df_processed.drop('contract', axis=1)

# Internet service
internet_dummies = pd.get_dummies(df_processed['internet_service'], prefix='internet')
df_processed = pd.concat([df_processed, internet_dummies], axis=1)
df_processed = df_processed.drop('internet_service', axis=1)

# Payment method
payment_dummies = pd.get_dummies(df_processed['payment_method'], prefix='payment')
df_processed = pd.concat([df_processed, payment_dummies], axis=1)
df_processed = df_processed.drop('payment_method', axis=1)

print("✓ Multi-category columns one-hot encoded")

# Handle online_security and tech_support (has 'No internet' option)
for col in ['online_security', 'tech_support']:
    df_processed[col] = df_processed[col].map({'Yes': 1, 'No': 0, 'No internet': 0})

print("✓ Service columns encoded")

# Drop the text churn_label, keep numeric churn
df_processed = df_processed.drop('churn_label', axis=1)

print("\n✓ All features are now numerical!")
print(f"\nFinal dataset shape: {df_processed.shape}")
print(f"Number of features: {df_processed.shape[1] - 1}")

In [ ]:
# Show the transformed data
print("Processed Dataset (first 5 rows):")
display(df_processed.head())

print("\nColumn names after encoding:")
print(list(df_processed.columns))

## Step 5: Prepare Data for Training

In [ ]:
# Separate features and target
X = df_processed.drop('churn', axis=1)
y = df_processed['churn']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

In [ ]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Maintain churn distribution
)

print("✓ Data split complete!\n")
print(f"Training samples: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing samples: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

print(f"\nChurn rate in training set: {y_train.mean()*100:.1f}%")
print(f"Churn rate in test set: {y_test.mean()*100:.1f}%")

## Step 6: Train Random Forest Classifier

**Random Forest** is an ensemble of Decision Trees - great for business data!

> **AI Assistant Prompt:**  
> "Train a Random Forest classifier with 100 trees on the churn dataset. Explain why Random Forest works well for customer churn prediction with mixed feature types."

In [ ]:
print("\n" + "="*60)
print("TRAINING RANDOM FOREST CLASSIFIER")
print("="*60)

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,      # 100 decision trees
    max_depth=10,          # Limit tree depth to prevent overfitting
    min_samples_split=20,  # Minimum samples to split a node
    random_state=42
)

print("\n[1/2] Training Random Forest (100 trees)...")
rf_model.fit(X_train, y_train)

print("[2/2] Making predictions...")
rf_predictions = rf_model.predict(X_test)
rf_probabilities = rf_model.predict_proba(X_test)[:, 1]

# Calculate accuracy
rf_accuracy = accuracy_score(y_test, rf_predictions)

print("\n✓ TRAINING COMPLETE!")
print("="*60)
print(f"\n🎯 Random Forest Accuracy: {rf_accuracy*100:.2f}%")

if rf_accuracy >= 0.80:
    print("\n🎉 EXCELLENT! Target accuracy achieved!")
elif rf_accuracy >= 0.75:
    print("\n👍 GOOD! Close to target.")
else:
    print("\n⚠️ Model could be better. Try tuning hyperparameters.")

In [ ]:
# Detailed classification report
print("Detailed Performance Metrics:")
print("="*60)
print(classification_report(
    y_test,
    rf_predictions,
    target_names=['No Churn', 'Churn']
))

print("\n📊 Key Insights:")
print("  • Precision (Churn): When we predict churn, how often are we right?")
print("  • Recall (Churn): Of customers who actually churned, how many did we catch?")
print("  • High recall is CRITICAL - we don't want to miss at-risk customers!")

In [ ]:
# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, rf_predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(cmap='Blues', ax=plt.gca())
plt.title('Confusion Matrix - Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nConfusion Matrix Breakdown:")
print(f"  • True Negatives (Correctly predicted No Churn): {cm[0,0]}")
print(f"  • False Positives (Predicted Churn, actually stayed): {cm[0,1]}")
print(f"  • False Negatives (Predicted No Churn, actually churned): {cm[1,0]} ⚠️")
print(f"  • True Positives (Correctly predicted Churn): {cm[1,1]}")

print("\n💰 Business Impact:")
print(f"  • False Negatives = {cm[1,0]} customers we thought would stay but left!")
print(f"  • False Positives = {cm[0,1]} customers we worried about unnecessarily")
print(f"  • Missing a churner is EXPENSIVE - lost revenue!")

## Step 7: Feature Importance Analysis

Which features are most important for predicting churn?

In [ ]:
# Get feature importance from Random Forest
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 10 Most Important Features for Churn Prediction:")
print("="*60)
display(feature_importance.head(10))

# Visualize top features
plt.figure(figsize=(12, 6))
top_features = feature_importance.head(10)
plt.barh(top_features['Feature'], top_features['Importance'], 
         color='#45B7D1', alpha=0.8, edgecolor='black')
plt.xlabel('Importance', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Top 10 Features for Churn Prediction', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Actionable Insights:")
print("  • Focus retention efforts on customers with high-importance risk factors")
print("  • Tenure is critical - first year is make-or-break!")
print("  • Contract type matters - encourage longer commitments")

## Step 8: Business Impact Analysis

Let's calculate the financial impact of our predictions.

**Assumptions:**
- Average customer lifetime value: $1,500
- Cost of retention campaign: $100 per customer
- Retention campaign success rate: 30%

In [ ]:
# Business parameters
avg_customer_value = 1500  # Average lifetime value
retention_cost = 100       # Cost to run retention campaign per customer
retention_success_rate = 0.30  # 30% of targeted customers stay

print("\n" + "="*70)
print("BUSINESS IMPACT ANALYSIS")
print("="*70)

# Get confusion matrix values
tn, fp, fn, tp = cm.ravel()

# Scenario 1: NO MODEL (do nothing)
actual_churners = (y_test == 1).sum()
loss_no_model = actual_churners * avg_customer_value

print("\nScenario 1: NO PREDICTION MODEL (Do Nothing)")
print(f"  Customers who churn: {actual_churners}")
print(f"  Lost revenue: ${loss_no_model:,}")

# Scenario 2: WITH MODEL (target predicted churners)
predicted_churners = (rf_predictions == 1).sum()
correctly_identified = tp  # True positives
saved_customers = int(correctly_identified * retention_success_rate)
saved_revenue = saved_customers * avg_customer_value
campaign_cost = predicted_churners * retention_cost
net_benefit = saved_revenue - campaign_cost

# Customers we missed
missed_churners = fn
lost_from_missed = missed_churners * avg_customer_value

print("\nScenario 2: WITH PREDICTION MODEL (Target At-Risk Customers)")
print(f"  Customers predicted to churn: {predicted_churners}")
print(f"  Correctly identified churners: {correctly_identified}")
print(f"  Customers saved (30% success rate): {saved_customers}")
print(f"  Revenue saved: ${saved_revenue:,}")
print(f"  Campaign cost: ${campaign_cost:,}")
print(f"  NET BENEFIT: ${net_benefit:,}")

print("\nMissed Opportunities:")
print(f"  Churners we missed: {missed_churners}")
print(f"  Lost revenue from missed: ${lost_from_missed:,}")

# Total impact
total_loss_with_model = lost_from_missed - net_benefit
improvement = loss_no_model - total_loss_with_model
improvement_pct = (improvement / loss_no_model) * 100

print("\n" + "="*70)
print("BOTTOM LINE")
print("="*70)
print(f"Total loss WITHOUT model: ${loss_no_model:,}")
print(f"Total loss WITH model: ${total_loss_with_model:,}")
print(f"\n💰 SAVINGS: ${improvement:,} ({improvement_pct:.1f}% improvement)")
print("\n✓ The ML model provides clear business value!")
print("="*70)

## Step 9: Compare with Other Algorithms

In [ ]:
print("\n" + "="*60)
print("COMPARING MULTIPLE ALGORITHMS")
print("="*60)

# Define models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
}

# Train and evaluate
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    results[name] = accuracy
    print(f"  ✓ Accuracy: {accuracy*100:.2f}%")

print("\n" + "="*60)

In [ ]:
# Visualize comparison
plt.figure(figsize=(10, 6))
models_list = list(results.keys())
accuracies = [results[m]*100 for m in models_list]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

bars = plt.bar(models_list, accuracies, color=colors, alpha=0.8, edgecolor='black')

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add target line
plt.axhline(y=80, color='red', linestyle='--', linewidth=2, label='Target (80%)')

plt.title('Algorithm Comparison - Churn Prediction', fontsize=14, fontweight='bold')
plt.xlabel('Algorithm', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.ylim([70, 100])
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Best model
best_model = max(results, key=results.get)
best_accuracy = results[best_model]

print(f"\n🏆 BEST MODEL: {best_model}")
print(f"   Accuracy: {best_accuracy*100:.2f}%")

## Step 10: Predict Churn for New Customers

In [ ]:
# Test with sample customers
print("\n" + "="*70)
print("PREDICTING CHURN FOR NEW CUSTOMERS")
print("="*70)

# Select 5 random test customers
sample_indices = np.random.choice(X_test.index, 5, replace=False)

for idx in sample_indices:
    customer_features = X_test.loc[idx:idx]
    actual_churn = y_test.loc[idx]
    
    # Predict
    prediction = rf_model.predict(customer_features)[0]
    probability = rf_model.predict_proba(customer_features)[0]
    
    # Get original customer data
    original_data = df.loc[idx]
    
    print(f"\n📋 Customer #{idx}:")
    print(f"   Tenure: {original_data['tenure_months']} months")
    print(f"   Contract: {original_data['contract']}")
    print(f"   Monthly charges: ${original_data['monthly_charges']:.2f}")
    print(f"   Tech support: {original_data['tech_support']}")
    print(f"\n   Prediction: {'🚨 WILL CHURN' if prediction==1 else '✓ WILL STAY'}")
    print(f"   Churn probability: {probability[1]*100:.1f}%")
    print(f"   Actual outcome: {'Churned' if actual_churn==1 else 'Stayed'}")
    print(f"   {'✓ CORRECT' if prediction==actual_churn else '✗ INCORRECT'}")
    
    if prediction == 1:
        print(f"   💡 Recommended action: Offer retention incentive (${retention_cost})")

print("\n" + "="*70)

## Step 11: Summary and Key Takeaways

In [ ]:
print("\n" + "="*70)
print("PROJECT SUMMARY - CUSTOMER CHURN PREDICTION")
print("="*70)

print("\n✅ WHAT YOU ACCOMPLISHED:")
print("  1. Created synthetic telecom customer dataset")
print("  2. Analyzed churn patterns and business drivers")
print("  3. Handled categorical variables (Label + One-Hot Encoding)")
print("  4. Trained Random Forest classifier")
print(f"  5. Achieved {rf_accuracy*100:.2f}% accuracy")
print("  6. Analyzed feature importance")
print("  7. Calculated business impact and ROI")
print("  8. Compared multiple algorithms")

print("\n🎯 KEY LEARNINGS:")
print("  • Feature engineering is critical for categorical data")
print("  • Random Forest works great for business data")
print("  • Feature importance reveals business insights")
print("  • For churn, RECALL matters more than precision")
print("  • ML models must provide business value")
print("  • False negatives (missed churners) are expensive!")

print("\n📊 FINAL RESULTS:")
print(f"  • Dataset: {len(df)} customers")
print(f"  • Churn rate: {df['churn'].mean()*100:.1f}%")
print(f"  • Features: {X.shape[1]} (after encoding)")
print(f"  • Training samples: {len(X_train)}")
print(f"  • Testing samples: {len(X_test)}")
print(f"  • Best model: {best_model}")
print(f"  • Best accuracy: {best_accuracy*100:.2f}%")
print(f"  • Target achieved: {'✅ YES' if best_accuracy >= 0.80 else '⚠️ CLOSE'}")
print(f"  • Business savings: ${improvement:,}")

print("\n💼 BUSINESS INSIGHTS:")
print("  • Month-to-month customers are HIGH RISK")
print("  • First 12 months are critical for retention")
print("  • Tech support reduces churn significantly")
print("  • Target at-risk customers with retention offers")

print("\n🚀 NEXT STEPS:")
print("  • Session 4: Multi-class classification (MNIST digits)")
print("  • Learn confusion matrix for 10 classes")
print("  • Understand per-class metrics")
print("  • Work with image data")

print("\n" + "="*70)
print("🎉 YOU BUILT A BUSINESS ML APPLICATION!")
print("="*70)

---

## AI Prompts for This Notebook

### Prompt 1: Dataset Creation
```
Create a synthetic telecom customer churn dataset with 1000 customers in Python. 
Include features: tenure, monthly charges, contract type (month-to-month, 1-year, 
2-year), internet service, tech support, and churn. Make churn correlate with short 
tenure and month-to-month contracts.
```

### Prompt 2: Feature Engineering
```
Show how to handle categorical variables in a churn dataset. Use Label Encoding for 
binary columns (Yes/No) and One-Hot Encoding for multi-category columns (contract 
type, payment method). Explain the difference between the two approaches.
```

### Prompt 3: Model Training
```
Train a Random Forest classifier with 100 trees for customer churn prediction. 
Split data 80/20 train/test. Calculate accuracy and show confusion matrix. 
Explain why Random Forest is good for this problem.
```

### Prompt 4: Feature Importance
```
Extract feature importance from a Random Forest churn model. Create a horizontal bar 
chart showing the top 10 most important features. Explain what this means for business 
strategy.
```

### Prompt 5: Business Impact
```
Calculate the financial impact of a churn prediction model. Assume average customer 
value is $1500, retention campaign costs $100, and has 30% success rate. Compare 
scenarios with and without the model. Show net savings.
```

### Prompt 6: Algorithm Comparison
```
Compare Random Forest, Decision Tree, and Logistic Regression for churn prediction 
on the same dataset. Create a bar chart comparing their accuracies. Which works best 
and why?
```

---

**End of Notebook**

**Created:** 2026-01-06  
**Course:** ML for Engineers - Module 2  
**Project:** Customer Churn Prediction  
**Target:** >80% Accuracy ✅